# 1. Preparación Inicial Dataframes

Este proyecto usó datos del [**"Online Shoppers Purchasing Intention Dataset"**](https://doi.org/10.24432/C5F88Q) creado por C. Sakar y Yomi Kastro (2018), alojado en el "UCI Machine Learning Repository". El dataset original tiene una licencia [**CC BY 4.0**](https://creativecommons.org/licenses/by/4.0/legalcode). Sobre los registros originales se aplicaron transformaciones de limpieza y preprocesamiento, particionado (*train/test*) y generación de variables derivadas. El código fuente de este proyecto se distribuye bajo la **Licencia MIT** (ver archivo `LICENSE`).

## 1.0. Importación de Librerías

In [1]:
#Importar librerias
import pandas as pd
import numpy as np
import joblib
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split

## 1.1. Exploración inicial de variables y formatos

In [ ]:
# fetch dataset from UCI repository
online_shoppers_purchasing_intention_dataset = fetch_ucirepo(id=468)
df_shoppers_raw = online_shoppers_purchasing_intention_dataset.data.original

# Alternativamente descargar el csv del repositorio
#df_shoppers_raw = pd.read_csv("online_shoppers_intention.csv", sep=",")

In [3]:
#Hacemos una copia sobre la que trabajar
df_shoppers = df_shoppers_raw.copy()
#Veamos información
df_shoppers.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType              12330 no

De entrada vemos que no hay ningún NA en el dataset. Lo cual es algo a tener en cuenta de cara a entrenar modelos, así como los pipelines de inputación y estadarización posteriores.

a) Describamos las variables numéricas:

In [4]:
#Veamos un describe de las numéricas
df_shoppers.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427,2.124006,2.357097,3.147364,4.069586
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917,0.911325,1.717277,2.401591,4.025169
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


b) Describamos las variables no numéricas:

In [5]:
#Veamos un describe de las no numéricas
df_shoppers.describe(exclude=['number'])

,Month,VisitorType,Weekend,Revenue
count,12330,12330,12330,12330
unique,10,3,2,2
top,May,Returning_Visitor,False,False
freq,3364,10551,9462,10422


In [6]:
#Veamos conteos por Month
df_shoppers.Month.value_counts()

Month
May     3364
Nov     2998
Mar     1907
Dec     1727
Oct      549
Sep      448
Aug      433
Jul      432
June     288
Feb      184
Name: count, dtype: int64

In [7]:
#Veamos conteos por VisitorType
df_shoppers.VisitorType.value_counts()

VisitorType
Returning_Visitor    10551
New_Visitor           1694
Other                   85
Name: count, dtype: int64

Por otro lado, vemos que es un dataset desbalanceado con respecto a la variable target (`Revenue`), 10422 de 12330 (84.53%) no compraron finalmente así que es algo a tener en cuenta también de cara al split.

c) Describamos ciertas variables numéricas como categóricas:

In [8]:
#Tratemos ciertas numericas como categóricas
cols_num_cat=['OperatingSystems', 'Browser', 'Region', 'TrafficType']
df_shoppers[cols_num_cat].astype('category').describe()

,OperatingSystems,Browser,Region,TrafficType
count,12330,12330,12330,12330
unique,8,13,9,20
top,2,2,1,2
freq,6601,7961,4780,3913


d) Veamos los valores únicos de todas las posibles variables categóricas y buleanas: 

In [9]:
#Posibles categoricas/boleanas
cols_cat = ['Month', 'VisitorType', 'Weekend', 'Revenue', 
            'OperatingSystems', 'Browser', 'Region', 'TrafficType']
resumen = {col: df_shoppers[col].explode().unique().tolist() for col in cols_cat}
print("\nResumen por columna:")
for key,vals in resumen.items():
    print(f"- {key}: {vals}")


Resumen por columna:
- Month: ['Feb', 'Mar', 'May', 'Oct', 'June', 'Jul', 'Aug', 'Nov', 'Sep', 'Dec']
- VisitorType: ['Returning_Visitor', 'New_Visitor', 'Other']
- Weekend: [False, True]
- Revenue: [False, True]
- OperatingSystems: [1, 2, 4, 3, 7, 6, 8, 5]
- Browser: [1, 2, 3, 4, 5, 6, 7, 10, 8, 9, 12, 13, 11]
- Region: [1, 9, 2, 3, 4, 5, 6, 7, 8]
- TrafficType: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 18, 19, 16, 17, 20]


Hay un total de 12330 entradas que según la descripción del dataset cada sesión corresponde a un único usuario en un periodo de un año para evitar tendencias a campañas específicas, días especiales, perfiles de usuarios, o periodos. Hay un total de 18 columnas:

- Las primeras 6 (Columnas 0-5) representan el número de página visitadas en la sesión para cada categoría de página (`Administrative`, `Informational` y `ProductRelated`) y el total de tiempo empleado en cada categoría (`Administrative_Duration`, `Informational_Duration`, `ProductRelated_Duration`). Los números de páginas son númericas discretas, mientras que los tiempos de duración son númericas continuas.

* Las siguientes 3 (Columnas 6-8) son métricas agregadas de "Google Analytics" que representan el valor promedio de los indicadores históricos de todas las páginas que el usuario visitó durante su sesión. Las 3 son variables numéricas continuas:

    - `BounceRates`: promedio de las tasas de rebote de todas las URLs visitadas en la sesión (parece estar en proporciones). Las tasas de rebote en Google Analytics, reporta el porcentaje de visitantes que acceden al sitio desde esa página y luego lo abandonan (rebotan) sin generar ninguna otra solicitud al servidor de análisis durante esa sesión.     

    * `ExitRates`: promedio de las tasas de salida de todas las URLs visitadas en la sesión (parece estar en proporciones). Las tasas de salida en Google Analytics, reporta el porcentaje de todas las visitas a dicha página que constituyeron la última visita de la sesión.
           
    - `PageValues`: promedio del valor monetario de todas las URLs visitadas en la sesión. El valor de las páginas en Google Analytics, reporta el valor monetario promedio de una página web que un usuario ha visitado antes de completar una transacción de comercio electrónico. Para nuestro objetivo, será conveniente eliminarla antes del modelado. Por cómo está construida, podría introducir un cierto *data leakage* y contaminar el *propensity score* que queremos generar más adelante. Las páginas finales del proceso de compra muy probablemente dispararán estos valores, generando una correlación muy fuerte con la variable target (`Revenue`). Esto no nos interesa, ya que nos gustaría que el modelo identificara usuarios que mostraron comportamientos similares a los compradores para futuras campañas de retargeting de los no compradores.
    
- `SpecialDay` (Columna 9): indica la proximidad de la fecha de visita al sitio web a un día especial, en el que es más probable que las sesiones concluyan en transacción. Este valor se determina teniendo en cuenta la dinámica del comercio electrónico, como el intervalo de tiempo entre la fecha del pedido y la fecha de entrega. Por ejemplo, para San Valentín, toma un valor distinto de cero entre el 2 y 12 de Febrero, es cero antes y después de esta fecha (a menos que se acerque a otra fecha especial) y su valor máximo de 1 se alcanza el 8 de Febrero. Por lo que es una variable numérica continua.

* `Month` (Columna 10): indica el mes en la que la sesión tuvo lugar. Como podemos ver solo se han representado 10 de los 12 meses posibles, por lo que es una limitación a tener en cuenta más adelante para tratar esta variable categórica. Además, los nombres no están estandarizados algunos aparecen con abreviaturas y otros con el nombre completo, así que tocará estandarizarlos.

- Columnas Codificadas (Columnas 11-14), tenemos `OperatingSystems`, `Browser`, `Region`, `TrafficType`, que como hemos visto arriba están codificadas así que carecen de significado interpretable, ya que no sabemos lo que significa cada categoría. Probablemente sea recomendable eliminarlas más adelante antes del modelado, ya que conservar variables que no podemos explicar puede dificultar la interpretación. De cara al análisis exploratorio vamos a conservarlas.

* `VisitorType` (Columna 15): indica el tipo de visitante como 'Returning_Visitor', 'New_Visitor', o 'Other'. Por lo que se trata de una variable categórica. 

- `Weekend` (Columna 16): indica si la visita tuvo lugar en fin de semana. Además, está como un buleano así que nos interesará pasarlo a binario.

* `Revenue` (Columna 17): indica si la visita acabo en una compra o no. Esta es la **variable target de interés**. Además, también está como un buleano así que nos interesará pasarlo a binario.

## 1.2. Limpieza inicial de variables

In [10]:
#Variable Month
##Definimos un dict para reemplazar los valores a los nombres completos de los 10 meses presentes
month_dict = {'Feb':'February', 'Mar':'March', 'May':'May','June':'June', 'Jul':'July', 
              'Aug':'August','Sep':'September','Oct':'October', 'Nov':'November', 'Dec':'December'}
df_shoppers["Month"] = df_shoppers["Month"].replace(month_dict)

#Variable Weekend (pasamos de buleano a binaria)
df_shoppers["Weekend"] = df_shoppers["Weekend"].astype(int)
#Variable Revenue (pasamos de buleano a binaria)
df_shoppers["Revenue"] = df_shoppers["Revenue"].astype(int)

In [11]:
df_shoppers.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  str    
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType              12330 no

Vemos que los cambios se han hecho efectivos.

In [12]:
#Describamos estas 3 variables como categoricas
df_shoppers[["Month","Weekend","Revenue"]].astype('category').describe()

,Month,Weekend,Revenue
count,12330,12330,12330
unique,10,2,2
top,May,0,0
freq,3364,9462,10422


In [13]:
#Veamos conteos por Month
df_shoppers.Month.value_counts()

Month
May          3364
November     2998
March        1907
December     1727
October       549
September     448
August        433
July          432
June          288
February      184
Name: count, dtype: int64

Como vemos los conteos concuerdan con los valores previos antes de la limpieza.

## 1.3. Derivación de variables
Vamos a generar también algunas variables derivadas de interés centradas en productos:

- **Proporción de Páginas de Producto** (`ProductRelated_Ratio`):

    * **Razonamiento**: para un problema de negocio nos puede interesar saber el total de las páginas relacionadas con productos (`ProductRelated`) con respecto al global de páginas visitadas. Así que tiene sentido ver cual es el ratio de páginas de producto con respecto al total de páginas.

    - **Cálculo**: `ProductRelated / total_pages` (el resultado deberá ser 0 cuando total_pages sea 0, para evitar dividir por cero, pero también considerar NaNs por valores faltantes), donde:

        * total_pages = `ProductRelated + Informational + Administrative`

* **Duración asociada a Productos** (`ProductRelated_Duration_Ratio`):

    - **Razonamiento**: para un problema de negocio también nos interesa saber el tiempo que pasa el usuario en páginas relacionadas con productos (`ProductRelated_Duration`) con respecto al tiempo global de la sesión. Así que tiene sentido ver cual es el ratio de tiempo que pasa en páginas de productos con respecto al tiempo total.

    * **Cálculo**: `ProductRelated_Duration / total_duration` (el resultado deberá ser 0 cuando total_duration sea 0, para evitar dividir por cero, pero también considerar NaNs por valores faltantes), donde:

        - total_duration = `ProductRelated_Duration + Informational_Duration + Administrative_Duration`

- **Duración por Página de Producto** (`Duration_Per_Product_Page`):

    * **Razonamiento**: para un problema de negocio también nos puede interesar fijarnos en cuanto tiempo a pasado el usuario por cada página asociada a productos.

    - **Cálculo**: `ProductRelated_Duration / ProductRelated` (el resultado deberá ser 0 cuando ProductRelated sea 0, para evitar dividir por cero, pero también considerar NaNs por valores faltantes).

En este caso, si que podemos realizar la derivación de variables antes del split train-test ya que no tenemos valores NA en ninguna de las variables asociadas originales. Tal y como se ha preparado el código si huviera algún NA en las variables originales se generaría un NA que posteriormente se podría imputar, lo que aportaría mayor control sobre que datos se imputan.

In [14]:
# 0. Calculamos los totales 
## Duración total de 3 categorías (total_duration) [esta forma de hacerlo propaga NAs, lo que queríamos]
total_duration = (df_shoppers["ProductRelated_Duration"] + df_shoppers["Informational_Duration"] + df_shoppers["Administrative_Duration"])
## Páginas totales de 3 categorías (total_pages) [esta forma de hacerlo propaga NAs, lo que queríamos]
total_pages = (df_shoppers["ProductRelated"] + df_shoppers["Informational"] + df_shoppers["Administrative"])

# 1. Calculamos Proporción de Páginas de Producto (ProductRelated_Ratio)
df_shoppers["ProductRelated_Ratio"] = np.where(total_pages.isna() | df_shoppers["ProductRelated"].isna(), np.nan, #Controlamos NAs generados por datos faltantes
    np.where(total_pages != 0, df_shoppers["ProductRelated"] / total_pages, 0) #Cuando la división sea entre cero devolvemos cero
    )

# 2. Calculamos Duración asociada a Productos (ProductRelated_Duration_Ratio)
df_shoppers["ProductRelated_Duration_Ratio"] = np.where(total_duration.isna() | df_shoppers["ProductRelated_Duration"].isna(), np.nan, #Controlamos NAs generados por datos faltantes
    np.where(total_duration != 0, df_shoppers["ProductRelated_Duration"] / total_duration, 0) #Cuando la división sea entre cero devolvemos cero
    )

# 3. Calculamos Duración por Página de Producto (Duration_Per_Product_Page)
df_shoppers["Duration_Per_Product_Page"] = np.where(df_shoppers["ProductRelated"].isna() | df_shoppers["ProductRelated_Duration"].isna(), np.nan, #Controlamos NAs generados por datos faltantes
    np.where(df_shoppers["ProductRelated"] != 0, df_shoppers["ProductRelated_Duration"] / df_shoppers["ProductRelated"], 0) #Cuando la división sea entre cero devolvemos cero
    )

In [15]:
df_shoppers.info()

<class 'pandas.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 21 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Administrative                 12330 non-null  int64  
 1   Administrative_Duration        12330 non-null  float64
 2   Informational                  12330 non-null  int64  
 3   Informational_Duration         12330 non-null  float64
 4   ProductRelated                 12330 non-null  int64  
 5   ProductRelated_Duration        12330 non-null  float64
 6   BounceRates                    12330 non-null  float64
 7   ExitRates                      12330 non-null  float64
 8   PageValues                     12330 non-null  float64
 9   SpecialDay                     12330 non-null  float64
 10  Month                          12330 non-null  str    
 11  OperatingSystems               12330 non-null  int64  
 12  Browser                        12330 non-null  int64  
 1

In [ ]:
#Guardamos el resultado en CSV
df_shoppers.to_csv("../data/interim/online_shoppers_intention_clean.csv", sep=",")

## 1.4. Split entre train y test sets

Ahora, vamos a realizar el split entre train y test sets, aplicaremos:

* Proporción 80-20 para train-test (`test_size=0.2`)

- Fijamos el random state (`random_state=42`)

* Estratificamos teniendo en cuenta la variable target, ya que hemos visto que estaba desbalanceada (`stratify=df_shoppers["Revenue"]`).

In [17]:
#Hacemos el split 
train_set, test_set = train_test_split(df_shoppers, test_size=0.2, random_state=42, stratify=df_shoppers["Revenue"])

In [ ]:
#Generamos un dict con los sets resultantes
df_shoppers_sets = {"train_set":train_set,"test_set":test_set}
#Guardamos en un joblib
joblib.dump(df_shoppers_sets, "../data/interim/df_shoppers_split.joblib", compress=3)

['df_shoppers_split.joblib']

**Nota:** Estos dos archivos son versiones intermedias que vamos a utilizar para realizar el EDA inicial sobre el train set antes de la selección de variables. Por ello hemos preparado una subcarpeta "interim" en data para guardar estos archivos intermedios. Más adelante en la fase de entrenamiento de los modelos les pasaremos la versión procesada definitiva generada por `src/data_processing.py` y el split equivalente generado por `src/training.py`.